In [1]:
class GoalBasedAgent:
    def __init__(self, goal, depth_limit=None):
        self.goal = goal
        self.depth_limit = depth_limit
    
    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal reached"
        return "Searching"
    
    def dls_search(self, graph, start, goal, depth_limit):
        visited = []  # List for visited nodes
        
        def dfs(node, depth):
            if depth > depth_limit:
                return None  # Limit reached
            
            visited.append(node)
            print(f"Visiting: {node} at depth {depth}")
            
            if node == goal:
                print(f"Goal {goal} found at depth {depth}!")
                return visited.copy()
            
            for neighbour in graph.get(node, []):
                if neighbour not in visited:
                    path = dfs(neighbour, depth + 1)
                    if path:
                        return path
            
            visited.pop()  # Backtrack if goal not found
            return None
        
        return dfs(start, 0)
    
    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        
        if goal_status == "Goal reached":
            return f"Goal {self.goal} already at current node!"
        else:
            if self.depth_limit is not None:
                print(f"Starting DLS from {percept} to {self.goal} with depth limit {self.depth_limit}")
                result = self.dls_search(graph, percept, self.goal, self.depth_limit)
                if result:
                    return f"Path to goal: {' -> '.join(result)}"
                else:
                    return f"Goal {self.goal} not found within depth limit {self.depth_limit}"
            else:
                return "No depth limit specified for DLS"

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': []
}

agent = GoalBasedAgent(goal='F', depth_limit=2)

result = agent.act('A', graph)
print(result)

agent2 = GoalBasedAgent(goal='F', depth_limit=3)
result2 = agent2.act('A', graph)
print(result2)

Starting DLS from A to F with depth limit 2
Visiting: A at depth 0
Visiting: B at depth 1
Visiting: D at depth 2
Visiting: E at depth 2
Visiting: C at depth 1
Visiting: F at depth 2
Goal F found at depth 2!
Path to goal: A -> C -> F
Starting DLS from A to F with depth limit 3
Visiting: A at depth 0
Visiting: B at depth 1
Visiting: D at depth 2
Visiting: E at depth 2
Visiting: F at depth 3
Goal F found at depth 3!
Path to goal: A -> B -> E -> F


In [2]:
class UtilityBasedAgent:
    def __init__(self, goal):
        self.goal = goal
    
    def compute_utility(self, cost):
        if cost == 0:
            return float('inf')
        return 1.0 / cost
    
    def ucs_search(self, graph, start, goal):
        frontier = [(start, 0)]  
        visited = set()  
        cost_so_far = {start: 0}  
        came_from = {start: None}  
        
        print(f"Starting UCS from {start} to {goal}")
        
        while frontier:
            frontier.sort(key=lambda x: x[1])
            current_node, current_cost = frontier.pop(0)
            if current_node in visited:
                continue
            
            print(f"Visiting: {current_node} with cost {current_cost}, utility: {self.compute_utility(current_cost):.4f}")
            visited.add(current_node)
            if current_node == goal:
                path = []
                node = current_node
                while node is not None:
                    path.append(node)
                    node = came_from[node]
                path.reverse()
                
                utility = self.compute_utility(current_cost)
                print(f"Goal {goal} found!")
                print(f"Path: {' -> '.join(path)}")
                print(f"Total Cost: {current_cost}")
                print(f"Total Utility: {utility:.4f}")
                return path, current_cost, utility
            
            for neighbor, cost in graph[current_node].items():
                new_cost = current_cost + cost
                
                if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                    cost_so_far[neighbor] = new_cost
                    came_from[neighbor] = current_node
                    frontier.append((neighbor, new_cost))  
        
        print(f"Goal {goal} not found")
        return None, None, None
    
    def act(self, percept, graph):
        print(f"Agent perceiving current node: {percept}")
        print(f"Agent goal: {self.goal}")
        
        if percept == self.goal:
            print("Goal already reached!")
            return f"Goal {self.goal} already at current node!"
        else:
            path, cost, utility = self.ucs_search(graph, percept, self.goal)
            if path:
                return f"Path found with cost {cost} and utility {utility:.4f}"
            else:
                return f"Goal {self.goal} not reachable from {percept}"
weighted_graph = {
    'A': {'B': 1, 'C': 4},
    'B': {'D': 2, 'E': 5},
    'C': {'F': 3},
    'D': {},
    'E': {'F': 1},
    'F': {}
}
agent = UtilityBasedAgent(goal='F')
result = agent.act('A', weighted_graph)
print(f"\nResult: {result}")
result2 = agent.act('B', weighted_graph)
print(f"\nResult: {result2}")

Agent perceiving current node: A
Agent goal: F
Starting UCS from A to F
Visiting: A with cost 0, utility: inf
Visiting: B with cost 1, utility: 1.0000
Visiting: D with cost 3, utility: 0.3333
Visiting: C with cost 4, utility: 0.2500
Visiting: E with cost 6, utility: 0.1667
Visiting: F with cost 7, utility: 0.1429
Goal F found!
Path: A -> C -> F
Total Cost: 7
Total Utility: 0.1429

Result: Path found with cost 7 and utility 0.1429
Agent perceiving current node: B
Agent goal: F
Starting UCS from B to F
Visiting: B with cost 0, utility: inf
Visiting: D with cost 2, utility: 0.5000
Visiting: E with cost 5, utility: 0.2000
Visiting: F with cost 6, utility: 0.1667
Goal F found!
Path: B -> E -> F
Total Cost: 6
Total Utility: 0.1667

Result: Path found with cost 6 and utility 0.1667


In [4]:
graph = {
    1: {2: 10, 3: 15, 4: 20},
    2: {1: 10, 3: 35, 4: 25},
    3: {1: 15, 2: 35, 4: 30},
    4: {1: 20, 2: 25, 3: 30}
}

def tsp_ucs(graph, start):
    frontier = [([start], 0)]
    min_cost = float('inf')
    best_path = None
    n = len(graph)

    while frontier:
        frontier.sort(key=lambda x: x[1])
        path, cost = frontier.pop(0)

        current = path[-1]
        if len(path) == n + 1 and path[-1] == start:
            if cost < min_cost:
                min_cost = cost
                best_path = path
            continue
        if len(path) == n:
            new_cost = cost + graph[current][start]
            frontier.append((path + [start], new_cost))
            continue
        for neighbor in graph[current]:
            if neighbor not in path:
                new_cost = cost + graph[current][neighbor]
                frontier.append((path + [neighbor], new_cost))

    return best_path, min_cost


start_city = 1
path, cost = tsp_ucs(graph, start_city)

print("Shortest Path:", path)
print("Minimum Cost:", cost)

Shortest Path: [1, 2, 4, 3, 1]
Minimum Cost: 80


In [3]:
tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E', 'A'],
    'C': ['F', 'G', 'A'],
    'D': ['H', 'B'],
    'E': ['F'],
    'F': ['I', 'C'],
    'G': [],
    'H': [],
    'I': []
}

def dls_tree(node, goal, depth, path):
    if depth == 0:
        return False
    if node == goal:
        path.append(node)
        return True
    if node not in tree:
        return False
    for child in tree[node]:
        if dls_tree(child, goal, depth - 1, path):
            path.append(node)
            return True
    return False

def dls_graph(node, goal, depth, path, visited):
    if depth == 0:
        return False
    if node == goal:
        path.append(node)
        return True
    if node not in graph:
        return False
    
    visited.add(node)
    for child in graph[node]:
        if child not in visited:
            if dls_graph(child, goal, depth - 1, path, visited):
                path.append(node)
                return True
    visited.remove(node)
    return False

def iterative_deepening_tree(start, goal, max_depth):
    for depth in range(max_depth + 1):
        print(f"Depth: {depth}")
        path = []
        if dls_tree(start, goal, depth, path):
            print("Path to goal:", " → ".join(reversed(path)))
            return
    print("Goal not found within depth limit.")

def iterative_deepening_graph(start, goal, max_depth):
    for depth in range(max_depth + 1):
        print(f"Depth: {depth}")
        path = []
        visited = set()
        if dls_graph(start, goal, depth, path, visited):
            print("Path to goal:", " → ".join(reversed(path)))
            return
    print("Goal not found within depth limit.")

print("Tree IDDFS:")
iterative_deepening_tree('A', 'I', 5)

print("\nGraph IDDFS:")
iterative_deepening_graph('A', 'I', 5)

Tree IDDFS:
Depth: 0
Depth: 1
Depth: 2
Depth: 3
Depth: 4
Path to goal: A → C → F → I

Graph IDDFS:
Depth: 0
Depth: 1
Depth: 2
Depth: 3
Depth: 4
Path to goal: A → C → F → I
